# Star Schema 
(SILVER >> GOLD)

# preparation (silver grouping support)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Catalog locations
SILVER_PATH = "`mvpde`.`2_silver`"

# Read Silver Tables (Assuming Silver stores cast/crew/genres as structured ARRAY<STRUCT>)
film_silver = spark.table(f"{SILVER_PATH}.film")
credit_silver = spark.table(f"{SILVER_PATH}.credit")

# Left join Film with supplementary Credit on movie_id
silver_combined = (
    film_silver.alias("f")
    .join(
        credit_silver.drop("title").alias("c"),
        col("f.id") == col("c.movie_id"),
        "left_outer"
    )
    .drop("movie_id")
)

(
    silver_combined
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_PATH}.film_combined")
)

# fact film 
Selects relevant features for fact, engineers new relevant columns for bayesian vote average (pondered rating), profit, roi, film historical Era and adds 1st production company as main

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Catalog locations
SILVER_PATH = "`mvpde`.`2_silver`"
GOLD_PATH = "`mvpde`.`3_gold`"

# Read Silver Table
silver_combined =  spark.table(f"{SILVER_PATH}.film_combined")

# Bayesian parameters: C = global mean rating, m = minimum votes threshold (90th percentile)
C_global = silver_combined.agg(F.avg("vote_average")).collect()[0][0] or 0.0
m_threshold = silver_combined.agg(F.percentile_approx("vote_count", 0.9)).collect()[0][0] or 0

# Expression for Bayesian formula: (v / (v + m) * R) + (m / (v + m) * C)
bayesian_expr = (
    (F.col("vote_count") / (F.col("vote_count") + F.lit(m_threshold))) * F.col("vote_average")
) + (
    (F.lit(m_threshold) / (F.col("vote_count") + F.lit(m_threshold))) * F.lit(C_global)
)

fact_film = (
    silver_combined
    .withColumn(
        # Extract name of the first company in array, or NULL if empty
        "primary_production_company",
        F.col("production_companies")[0]
    )
    .withColumn("profit", F.col("revenue") - F.col("budget"))
    .withColumn(
        "roi", 
        F.when(F.col("budget") > 0, F.round(F.col("revenue") / F.col("budget"), 2))
        .otherwise(None)
    )
    .select(
        F.col("id"),
        F.col("title"),
        F.col("original_title"),
        F.col("release_date"),
        F.year(F.to_date(F.col("release_date"))).alias("release_year"),
        F.col("status"),
        F.col("original_language"),
        F.col("budget"),
        F.col("revenue"),
        F.col("profit"),
        F.col("roi"),
        F.col("primary_production_company"),
        F.col("runtime"),
        F.col("popularity"),
        F.col("vote_average"),
        F.col("vote_count"),
        F.col("adult"),
        
        # Cinema Era
        F.when(F.year(F.to_date(F.col("release_date"))) < 1927, "Silent Era")
         .when((F.year(F.to_date(F.col("release_date"))) >= 1927) & (F.year(F.to_date(F.col("release_date"))) <= 1968), "Golden Age of Hollywood")
         .when((F.year(F.to_date(F.col("release_date"))) >= 1969) & (F.year(F.to_date(F.col("release_date"))) <= 1979), "New Hollywood")
         .when((F.year(F.to_date(F.col("release_date"))) >= 1980) & (F.year(F.to_date(F.col("release_date"))) <= 1999), "Blockbuster Era")
         .when((F.year(F.to_date(F.col("release_date"))) >= 2000) & (F.year(F.to_date(F.col("release_date"))) <= 2019), "Digital & Franchise Era")
         .when(F.year(F.to_date(F.col("release_date"))) >= 2020, "Streaming & Post-Pandemic Era")
         .otherwise("Unknown").alias("film_era")
    )
    .filter(F.col("id").isNotNull())
    .dropDuplicates(["id"])
    
    # Safely guard against zero/null denominator
    .withColumn(
        "bayesian_rating",
        F.when(
            F.col("vote_count").isNull() | (F.col("vote_count") == 0), 
            F.lit(None).cast("double")  # Returns NULL for unrated / 0-vote movies
        ).otherwise(F.round(bayesian_expr, 2))
    )
)

fact_film.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_PATH}.fact_film")


In [0]:
%sql
-- =========================================================================
-- Unity Catalog Column Comments for Gold Film Fact Table
-- =========================================================================

-- 1. Temporal & Categorical Attributes
ALTER TABLE mvpde.`3_gold`.fact_film ALTER COLUMN release_year 
COMMENT 'INT: Four-digit calendar year extracted from release_date (e.g., 1999).';

ALTER TABLE mvpde.`3_gold`.fact_film ALTER COLUMN film_era 
COMMENT 'STRING: Historical era classification based on release_year (e.g., "Silent Era" [<1927], "Golden Age" [1927-1968], "New Hollywood" [1969-1979], "Blockbuster Era" [1980-1999], "Modern Era" [2000+]).';

ALTER TABLE mvpde.`3_gold`.fact_film ALTER COLUMN primary_production_company 
COMMENT 'STRING: Lead studio or main production company extracted from the first element of production_companies array.';

-- 2. Financial Metrics
ALTER TABLE mvpde.`3_gold`.fact_film ALTER COLUMN profit 
COMMENT 'BIGINT: Net financial yield in USD calculated as (revenue - budget). NULL if either budget or revenue is missing or zero.';

ALTER TABLE mvpde.`3_gold`.fact_film ALTER COLUMN roi 
COMMENT 'DOUBLE: Return on Investment metric calculated as (revenue - budget) / budget. Expressed as a multiplier (e.g., 2.5 = 250% ROI). NULL when budget is missing or zero.';

-- 3. Advanced Rating Metrics
ALTER TABLE mvpde.`3_gold`.fact_film ALTER COLUMN bayesian_rating 
COMMENT 'DOUBLE: Weighted score computed using a Bayesian average formula [WR = (v/(v+m))*R + (m/(v+m))*C]. Adjusts raw vote averages against total vote counts to prevent low-vote bias (IMDb top 250 style).';

# genres
Finishes parsing genre, divides into dim_genres (short list and id pair) and bridge_film_genre (ref table for multiples)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Catalog locations
SILVER_PATH = "`mvpde`.`2_silver`"
GOLD_PATH = "`mvpde`.`3_gold`"

# Read Silver Table
silver_combined =  spark.table(f"{SILVER_PATH}.film_combined")

# 2. Use posexplode to capture the zero-indexed position (0 = main genre)
genres_exploded = (
    silver_combined
    .select(
        F.col("id"),
        F.posexplode_outer("genres").alias("genre_position", "genre_name")
    )
    .filter(F.col("genre_name").isNotNull())
)

# ------------------------------------------------------------------
# STEP 3: Generate gold_dim_genres (Unique Dimension)
# ------------------------------------------------------------------
# Generate surrogate genre_id using xxhash64 on genre_name
dim_genres = (
    genres_exploded
    .select("genre_name")
    .distinct()
    .withColumn("genre_id", F.abs(F.xxhash64(F.col("genre_name"))))
    .select("genre_id", "genre_name")
)

# Save Master Dimension
(
    dim_genres
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_PATH}.dim_genres")
)

# ------------------------------------------------------------------
# STEP 4: Generate gold_bridge_movie_genres (Bridge Table with Position)
# ------------------------------------------------------------------
bridge_film_genres = (
    genres_exploded
    .join(dim_genres, on="genre_name", how="inner")
    .select(
        F.col("id"),
        F.col("genre_id"),
        F.col("genre_position"),
        # Flag position 0 as the primary genre for quick filtering
        (F.col("genre_position") == 0).alias("is_primary_genre")
    )
    .distinct()
)

# Save Bridge Table
(
    bridge_film_genres
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_PATH}.bridge_film_genres")
)



# team (cast & crew)
Finishes parsing cast and crew, using a common dim_people table to avoid duplicates and bridge_film_cast and bridge_film_crew to administer connections

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Catalog locations
SILVER_PATH = "`mvpde`.`2_silver`"
GOLD_PATH = "`mvpde`.`3_gold`"

# Read Silver Table
silver_combined =  spark.table(f"{SILVER_PATH}.film_combined")

# ------------------------------------------------------------------
# STEP 1: Explode Cast and Crew Arrays
# ------------------------------------------------------------------
cast_exploded = (
    silver_combined
    .filter(F.col("cast").isNotNull())
    .select(
        F.col("id"),
        F.explode(F.col("cast")).alias("c")
    )
)

crew_exploded = (
    silver_combined
    .filter(F.col("crew").isNotNull())
    .select(
        F.col("id"),
        F.explode(F.col("crew")).alias("cr")
    )
)

# ------------------------------------------------------------------
# STEP 2: Unified gold_dim_people (Combines Cast & Crew Individuals)
# ------------------------------------------------------------------
cast_people = cast_exploded.select(
    F.col("c.id").cast("long").alias("person_id"),
    F.col("c.name").alias("person_name"),
    F.when(F.col("c.gender") == 1, "Female")
     .when(F.col("c.gender") == 2, "Male")
     .when(F.col("c.gender") == 3, "Non-Binary")
     .otherwise("Unspecified").alias("gender")
)

crew_people = crew_exploded.select(
    F.col("cr.id").cast("long").alias("person_id"),
    F.col("cr.name").alias("person_name"),
    F.when(F.col("cr.gender") == 1, "Female")
     .when(F.col("cr.gender") == 2, "Male")
     .when(F.col("cr.gender") == 3, "Non-Binary")
     .otherwise("Unspecified").alias("gender")
)

# Union cast and crew people, deduplicating on person_id
dim_people = (
    cast_people.union(crew_people)
    .filter(F.col("person_id").isNotNull())
    .dropDuplicates(["person_id"])
)

# Save Shared Dimension Table
(
    dim_people
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_PATH}.dim_people")
)

# ------------------------------------------------------------------
# STEP 3: Generate bridge_film_crew
# ------------------------------------------------------------------
bridge_film_crew = (
    crew_exploded
    .select(
        F.col("id"),
        F.col("cr.id").cast("long").alias("person_id"),
        F.col("cr.department").alias("department"),
        F.col("cr.job").alias("job"),
        F.col("cr.credit_id").alias("credit_id")
    )
    .filter(F.col("person_id").isNotNull() & F.col("id").isNotNull())
    .distinct()
)

# Save Crew Bridge Table
(
    bridge_film_crew
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_PATH}.bridge_film_crew")
)


# ------------------------------------------------------------------
# Generate bridge_film_cast (Relationship Bridge Table)
# ------------------------------------------------------------------
bridge_film_cast = (
    cast_exploded
    .select(
        F.col("id"),
        F.col("c.id").cast("long").alias("person_id"),
        F.col("c.character").alias("character_name"),
        F.col("c.order").cast("integer").alias("cast_order"),
        F.col("c.cast_id").cast("integer").alias("cast_id"),
        F.col("c.credit_id").alias("credit_id")
    )
    .filter(F.col("person_id").isNotNull() & F.col("id").isNotNull())
    .distinct()
)

# Persist Bridge to Delta Lake
(
    bridge_film_cast
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_PATH}.bridge_film_cast")
)


In [0]:
%sql
-- =========================================================================
-- 1. Table-Level Metadata & Governance Tags
-- =========================================================================
ALTER TABLE mvpde.`3_gold`.bridge_film_cast 
SET TAGS (
  'layer' = 'gold', 
  'entity_type' = 'bridge', 
  'domain' = 'films'
);

COMMENT ON TABLE mvpde.`3_gold`.bridge_film_cast IS 
'Gold layer bridge table resolving the many-to-many (N:M) relationship between movies and cast members. Contains role metadata and billing order.';

-- =========================================================================
-- 2. Foreign Keys & Core Identifiers
-- =========================================================================
ALTER TABLE mvpde.`3_gold`.bridge_film_cast ALTER COLUMN id 
COMMENT 'BIGINT: Unique TMDB movie identifier, foreign key linking to mvpde.3_gold.fact_movie.';

ALTER TABLE mvpde.`3_gold`.bridge_film_cast ALTER COLUMN person_id 
COMMENT 'BIGINT: Unique TMDB actor/person identifier, foreign key linking to mvpde.3_gold.dim_person.';

-- =========================================================================
-- 3. Role & Credit Attributes
-- =========================================================================
ALTER TABLE mvpde.`3_gold`.bridge_film_cast ALTER COLUMN character_name 
COMMENT 'STRING: Name of the fictional or real-life character portrayed by the actor in the film.';

ALTER TABLE mvpde.`3_gold`.bridge_film_cast ALTER COLUMN cast_order 
COMMENT 'INT: Billing order index of the actor in the movie credits (0 represents top/lead billing).';

ALTER TABLE mvpde.`3_gold`.bridge_film_cast ALTER COLUMN cast_id 
COMMENT 'INT: TMDB internal production cast tracking identifier.';

ALTER TABLE mvpde.`3_gold`.bridge_film_cast ALTER COLUMN credit_id 
COMMENT 'STRING: Unique alphanumeric alphanumeric hash key for the specific credit record.';


# extra consult checks

In [0]:
%sql
SELECT 
    f.title,
    b.cast_order,
    p.person_name,
    b.character_name,
    p.gender
FROM mvpde.`3_gold`.fact_film f
INNER JOIN mvpde.`3_gold`.bridge_film_cast b 
    ON f.id = b.id
INNER JOIN mvpde.`3_gold`.dim_people p 
    ON b.person_id = p.person_id
WHERE f.id = 14
ORDER BY b.cast_order ASC;

In [0]:
%sql
CREATE OR REPLACE VIEW mvpde.`3_gold`.vw_film_summary AS

WITH primary_genre AS (
    -- 1. Filter bridge for primary genre (position = 0 or flag)
    SELECT 
        bg.id,
        dg.genre_name AS primary_genre
    FROM mvpde.`3_gold`.bridge_film_genres bg
    INNER JOIN mvpde.`3_gold`.dim_genres dg 
        ON bg.genre_id = dg.genre_id
    WHERE bg.is_primary_genre = TRUE
),

movie_directors AS (
    -- 2. Extract directors from crew bridge
    SELECT 
        bc.id,
        -- Combines co-directors if multiple exist for a single movie
        ARRAY_JOIN(ARRAY_AGG(p.person_name), ', ') AS directors
    FROM mvpde.`3_gold`.bridge_film_crew bc
    INNER JOIN mvpde.`3_gold`.dim_people p 
        ON bc.person_id = p.person_id
    WHERE bc.job = 'Director'
    GROUP BY bc.id
),

top_3_cast AS (
    -- 3. Extract top 3 billed cast members ordered by cast_order
    SELECT 
        bc.id,
        ARRAY_JOIN(
            SLICE(
                TRANSFORM(ARRAY_SORT(ARRAY_AGG(STRUCT(bc.cast_order, p.person_name))), x -> x.person_name), 
                1, 3
            ), 
            ', '
        ) AS top_3_cast
    FROM mvpde.`3_gold`.bridge_film_cast bc
    INNER JOIN mvpde.`3_gold`.dim_people p 
        ON bc.person_id = p.person_id
    WHERE bc.cast_order < 3
    GROUP BY bc.id
)

-- 4. Main Query: Join Fact Table with pre-aggregated dimensional CTEs
SELECT 
    f.id,
    f.title,
    f.release_date,
    f.status,
    f.original_language,
    f.budget,
    f.revenue,
    f.runtime,
    f.popularity,
    f.vote_average,
    f.vote_count,
    COALESCE(pg.primary_genre, 'Unassigned') AS primary_genre,
    COALESCE(md.directors, 'Unknown') AS director,
    COALESCE(tc.top_3_cast, 'None Listed') AS top_3_cast
FROM mvpde.`3_gold`.fact_film f
LEFT JOIN primary_genre pg ON f.id = pg.id
LEFT JOIN movie_directors md ON f.id = md.id
LEFT JOIN top_3_cast tc ON f.id = tc.id;


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, coalesce, lit, when

SILVER_PATH = "`mvpde`.`2_silver`"
credit_silver = spark.table(f"{SILVER_PATH}.credit")
credit_silver.select("cast").printSchema()
credit_silver.select("crew").printSchema()

In [0]:
%sql
SELECT f.id, f.title, TRANSFORM(ARRAY_SORT(ARRAY_AGG(STRUCT(b.genre_position, g.genre_name))), x -> x.genre_name) AS genres FROM mvpde.`3_gold`.fact_film f LEFT JOIN mvpde.`3_gold`.bridge_film_genres b ON f.id = b.id LEFT JOIN mvpde.`3_gold`.dim_genres g ON b.genre_id = g.genre_id WHERE f.id = 101 GROUP BY f.id, f.title; 


In [0]:
%sql
SELECT
    AVG(vote_average)      AS avg_vote_average,
    AVG(bayesian_rating)   AS avg_bayesian_rating
FROM mvpde.`3_gold`.fact_film;